In [26]:
import pandas as pd
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

jobs = pd.read_csv("../data/exports/jobs.csv")
job_skills = pd.read_csv("../data/exports/job_skills.csv")
skill_counts = pd.read_csv("../data/exports/skill_counts.csv")
employer_counts = pd.read_csv("../data/exports/employer_counts.csv")
location_counts = pd.read_csv("../data/exports/location_counts.csv")
workplace_type_counts = pd.read_csv("../data/exports/workplace_type_counts.csv")
experience_counts = pd.read_csv("../data/exports/experience_counts.csv")

jobs.head()

,job_id,job_title,employer,municipality,region,publication_date,search_term,experience_required,workplace_type,years_experience
0,31206914,Data Engineer,SVERIGES RIKSBANK,Stockholm,Stockholms län,2026-06-23T14:13:36.000Z,data engineer,True,remote,2.0
1,31083431,Data Engineer,NXT Interim AB,Stockholm,Stockholms län,2026-05-26T11:19:55.000Z,data engineer,True,remote,5.0
2,30979729,Data Engineer,Techrytera AB,Jönköping,Jönköpings län,2026-04-30T11:53:20.000Z,data engineer,True,not_specified,NaN
3,30708777,Data Engineer,CoreChange Tech AB,Stockholm,Stockholms län,2026-03-06T14:52:40.000Z,data engineer,True,not_specified,NaN
4,31209863,Data Engineer,RebTel Networks AB,Stockholm,Stockholms län,2026-06-24T10:31:28.000Z,data engineer,True,not_specified,3.0


In [27]:
jobs.columns.tolist()

['job_id',
 'job_title',
 'employer',
 'municipality',
 'region',
 'publication_date',
 'search_term',
 'experience_required',
 'workplace_type',
 'years_experience']

In [28]:
jobs.dtypes

job_id                   int64
job_title                  str
employer                   str
municipality               str
region                     str
publication_date           str
search_term                str
experience_required       bool
workplace_type             str
years_experience       float64
dtype: object

In [29]:
from assistant.tools import get_top_skills

print(get_top_skills(role="Data Engineer"))

     skill  count
       sql     71
    python     69
     azure     51
       aws     42
databricks     41
       dbt     37
 snowflake     34
       gcp     34
  power bi     28
   airflow     24


In [30]:
print(get_top_skills())

     skill  count
    python    307
       sql    149
     azure    129
       aws    119
       gcp     86
databricks     57
 snowflake     46
       dbt     45
  power bi     45
   airflow     31


In [31]:
from assistant.tools import get_top_employers

print(get_top_employers(skill="python"))

                        employer  count
               Intensogruppen AB     10
                   Techrytera AB      9
Capgemini Engineering Sverige AB      8
         ACADEMIC WORK SWEDEN AB      7
         Eccera Professionals AB      7
                       Avaron AB      6
     KUNGLIGA TEKNISKA HÖGSKOLAN      6
                Knowit AB (Publ)      6
                      Veritaz AB      6
                  Friday Väst AB      5


In [32]:
from assistant.tools import get_top_locations

print(get_top_locations())

              region municipality  count
      Stockholms län    Stockholm    215
Västra Götalands län     Göteborg     90
           Skåne län         Lund     27
           Skåne län        Malmö     27
      Stockholms län        Solna     16
         Uppsala län      Uppsala     15
   Östergötlands län    Linköping     15
      Jönköpings län    Jönköping      9
   Västerbottens län         Umeå      6
   Östergötlands län   Norrköping      6


In [33]:
from assistant.tools import get_workplace_type_distribution

print(get_workplace_type_distribution())

workplace_type  count
 not_specified    411
        hybrid     84
        remote     27


In [34]:
from assistant.tools import get_experience_distribution

print(get_experience_distribution(role="data engineer"))

Note: 64 of 93 matching jobs had no explicit years-of-experience requirement mentioned and are excluded below.

 years_experience  count
              2.0      3
              3.0     10
              4.0      3
              5.0     12
              7.0      1


In [35]:
from assistant.tools import _load_jobs

jobs_df = _load_jobs()
print(jobs_df["years_experience"].isna().sum())
print(len(jobs_df))

364
522


In [36]:
from assistant.agent import agent

response = agent.invoke({"messages": [("user", "What are the top skills for data engineer roles?")]})
print(response["messages"][-1].content)

The top skills for data engineer roles are: SQL (71 mentions), Python (69 mentions), Azure (51 mentions), AWS (42 mentions), Databricks (41 mentions), dbt (37 mentions), Snowflake (34 mentions), GCP (34 mentions), Power BI (28 mentions), and Airflow (24 mentions).


In [37]:
print(agent.invoke({"messages": [("user", "Which employers in Stockholm mention Azure most often?")]})["messages"][-1].content)

Here are the top employers in Stockholm that mention Azure most often in their job postings, based on the collected dataset:

*   Bannerflow AB (4 postings)
*   Kraftsam Rekrytering & Bemanning AB (3 postings)
*   ACADEMIC WORK SWEDEN AB (2 postings)
*   Lynqa AB (2 postings)
*   Knowit AB (Publ) (2 postings)
*   Redeploy AB (2 postings)
*   UU Brand & Recruit AB (2 postings)
*   Viaplay Group Sweden AB (2 postings)
*   Quest Consulting Sverige AB (2 postings)
*   NOBA Bank Group AB (publ) (2 postings)

Please note that this reflects a snapshot of collected job postings, not real-time current openings.


In [38]:
print(agent.invoke({"messages": [("user", "What experience level is typical for machine learning roles?")]})["messages"][-1].content)

For machine learning roles, based on the job postings that explicitly mentioned a requirement, 3 years of experience appears twice, 5 years appears twice, and 4 and 10 years of experience appear once each.

It's important to note that 7 out of 13 matching machine learning job postings did not specify an explicit years-of-experience requirement and are therefore not included in this breakdown.


In [40]:
response = agent.invoke({"messages": [("user", "What are the top skills for data engineer roles?")]})
for msg in response["messages"]:
    print(type(msg).__name__, "-", msg.content[:200] if msg.content else "(tool call)")

HumanMessage - What are the top skills for data engineer roles?
AIMessage - (tool call)
ToolMessage -      skill  count
       sql     71
    python     69
     azure     51
       aws     42
databricks     41
       dbt     37
 snowflake     34
       gcp     34
  power bi     28
   airflow     24
AIMessage - The top skills for Data Engineer roles are: SQL (71 postings), Python (69 postings), Azure (51 postings), AWS (42 postings), Databricks (41 postings), dbt (37 postings), Snowflake (34 postings), GCP (
